# YOLO + PARSeq Smoke Integration

Bukti sambungan detektor (`ml/deteksi-plat` YOLOv8n) dengan OCR (`ml/ocr` PARSeq).

Flow: full frame -> YOLO `best_detector.pt` -> crop + pad 8% -> PARSeq `parseq_plate_best.pt` -> `{text, confidence, valid_format}`.

Repo di-clone dari GitHub (publik) agar bobot + kode integrasi yang dipakai persis versi repo.

In [ ]:
import os
REPO = "/kaggle/working/PBL-SmartPark"
if not os.path.isdir(REPO):
    print("cloning repo...")
    os.system("git clone --depth 1 https://github.com/adityarizqiqa/PBL-SmartPark.git " + REPO)
else:
    print("repo sudah ada, pull terbaru...")
    os.system(f"git -C {REPO} pull --ff-only")

DET = f"{REPO}/ml/deteksi-plat/fase8_export_detector/output/best_detector.pt"
OCR = f"{REPO}/ml/ocr/checkpoints/parseq_plate_best.pt"
for p in (DET, OCR):
    print(("OK  " if os.path.exists(p) else "HILANG "), f"{os.path.getsize(p)/2**20:.1f} MB", p)

In [ ]:
import sys
sys.path.insert(0, f"{REPO}/ml/ocr/scripts/benchmark")
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
try:
    import ultralytics
    print("ultralytics:", ultralytics.__version__)
except ImportError:
    print("install ultralytics...")
    os.system("pip install -q ultralytics")
    import ultralytics
    print("ultralytics:", ultralytics.__version__)

In [ ]:
from ultralytics import YOLO
from predict import load_ocr, predict_from_yolo_result

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
detector = YOLO(DET)
ocr = load_ocr(OCR, device=DEVICE)
print("models loaded")

In [ ]:
import glob
cands = sorted(glob.glob("/kaggle/input/**/*.jpg", recursive=True) + glob.glob("/kaggle/input/**/*.png", recursive=True) + glob.glob("/kaggle/input/**/*.jpeg", recursive=True))
print(f"kandidat gambar di /kaggle/input: {len(cands)}")
for c in cands[:3]:
    print(" -", c)
LIMIT = 10
files = cands[:LIMIT]

In [ ]:
import cv2
import pandas as pd

CONF = 0.25
rows, n_ok = [], 0
for fp in files:
    img = cv2.imread(fp, cv2.IMREAD_COLOR)
    if img is None:
        print(f"SKIP (unreadable): {fp}")
        continue
    res = detector.predict(img, conf=CONF, imgsz=640, verbose=False)[0]
    plates = predict_from_yolo_result(ocr, res, image=img, conf_threshold=CONF)
    nbox = len(res.boxes) if res.boxes is not None else 0
    shown = ", ".join(f"{r['text']}({r['confidence']:.2f})" for r in plates) or "-"
    print(f"{fp.split('/')[-1][:45]:45s} boxes={nbox} -> {shown}")
    for r in plates:
        rows.append({"file": fp.split("/")[-1], "text": r["text"], "confidence": r["confidence"],
                       "valid_format": r["valid_format"], "det_conf": r["det_conf"], "box": r["box"]})
        n_ok += bool(r["valid_format"])

df = pd.DataFrame(rows)
print(f"\nSMOKE RESULT: {len(files)} images, {len(rows)} plate crops, {n_ok} valid format")
print("PASS" if len(rows) > 0 else "WARN: detector produced no boxes (cek --conf / gambar)")
df.head(20)

In [ ]:
df.to_csv("/kaggle/working/yolo_parseq_smoke_results.csv", index=False)
print("saved: /kaggle/working/yolo_parseq_smoke_results.csv")
print(df.groupby("valid_format").size() if len(df) else "(no rows)")